# Tutorial 07: Advanced Training

**Time**: 45 minutes | **Difficulty**: Advanced | **Prerequisites**: Tutorials 01-06

---

## What You'll Learn
- Curriculum learning: gradually increase difficulty as the agent improves
- Multi-seed training: why a single run can mislead you
- Distributed training: run many agents in parallel with Multiverse
- Safety: what Multiverse's sentinel and safety gates do

---

## The Core Problem with Single Runs

A single training run is not reliable evidence.

```
Seed 42:  final return = 0.91  ← great!
Seed 7:   final return = 0.23  ← terrible!
Seed 13:  final return = 0.78
```

RL is stochastic. The same algorithm with the same config can succeed or fail
depending on initial random choices. You need multiple seeds to know if your
algorithm actually works.

---

## Setup

In [ ]:
import subprocess
import json
import pathlib

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    output = result.stdout + result.stderr
    print(output if output.strip() else '(no output)')
    return result.returncode == 0

run('multiverse doctor')

## Part 1: Multi-Seed Training

Run the same config across three seeds. Compare the spread of results.
This is the minimum bar for trusting a result.

In [ ]:
seeds = [42, 7, 13]

for seed in seeds:
    print(f'--- Seed {seed} ---')
    run(f'multiverse train --algo q --verse grid_world --episodes 150 --seed {seed}')
    print()

In [ ]:
# List all runs so we can compare
print('All recent runs:')
run('multiverse runs list')

## Part 2: Curriculum Learning

Curriculum learning trains on progressively harder versions of a task.

**Without curriculum**: Agent is thrown into the hard task immediately.
Often the reward signal is too sparse to learn from at all.

**With curriculum**:
```
Stage 1: line_world (trivial) → agent learns basic navigation
Stage 2: grid_world (easy)    → agent applies spatial reasoning
Stage 3: maze_world (hard)    → agent builds on prior skills
```

This mirrors how humans learn: walk before you run.

In [ ]:
# Curriculum stage 1: master the easy task first
print('=== Curriculum Stage 1: line_world ===')
run('multiverse train --algo q --verse line_world --episodes 100 --seed 42')

In [ ]:
# Curriculum stage 2: move to a harder task
print('=== Curriculum Stage 2: grid_world ===')
run('multiverse train --algo q --verse grid_world --episodes 150 --seed 42')

In [ ]:
# Curriculum stage 3: the hard task
print('=== Curriculum Stage 3: maze_world ===')
run('multiverse train --algo q --verse maze_world --episodes 200 --seed 42')

In [ ]:
# Compare: try maze_world without the curriculum (cold start)
print('=== Cold start: maze_world without curriculum ===')
run('multiverse train --algo q --verse maze_world --episodes 200 --seed 99')

**Compare the maze_world results**: the curriculum run (seed 42) should show faster early improvement than the cold start (seed 99), because prior training built useful representations.

## Part 3: Distributed Training

Multiverse's `dist` command runs many agents in parallel — useful for:
- Running multi-seed experiments automatically
- Population-based training (PBT): agents share and compete
- Covering hyperparameter space faster

In [ ]:
# See what distributed training looks like
print('Distributed training options:')
run('multiverse dist --help')

In [ ]:
# Run a small sharded job: 3 seeds in parallel
print('Sharded distributed run (3 workers):')
run('multiverse dist --mode sharded --verse grid_world --algo q -- --n_workers 3 --episodes 100')

## Part 4: Safety and the Sentinel

Multiverse includes a **sentinel** that monitors training for unsafe or degraded behaviour:

- Detects performance collapse (returns drop sharply)
- Detects stuck agents (returns plateau for too long)
- Can halt or rollback training automatically

This matters for production deployments where you can't manually monitor every run.

In [ ]:
# Check sentinel status
print('Sentinel / safety system status:')
run('multiverse status')

In [ ]:
# Inspect what artifacts a typical run produces (policy, metrics, safety log)
print('Run artifacts inspection:')
run('multiverse runs latest')
print()
run('multiverse runs files latest')

## Part 5: Reading Run Artifacts Directly

Every training run saves metrics to its run directory.
You can load and analyze these directly in Python.

In [ ]:
import json
import pathlib
import glob

# Find the most recent run directory
runs_root = pathlib.Path('runs')
run_dirs = sorted(runs_root.glob('*'), key=lambda p: p.stat().st_mtime, reverse=True)

if run_dirs:
    latest = run_dirs[0]
    print(f'Latest run: {latest}')
    print()
    
    # Try to read metrics
    for fname in ('metrics.json', 'episode_returns.json', 'summary.json', 'train_log.jsonl'):
        p = latest / fname
        if p.exists():
            with open(p) as f:
                content = f.read()
            print(f'--- {fname} (first 500 chars) ---')
            print(content[:500])
            print()
else:
    print('No runs found. Run some training first!')

## Advanced Training Checklist

Before trusting a result in production:

- [ ] Run at least 3 seeds — check mean AND variance
- [ ] Use curriculum if the task is hard (sparse reward, large space)
- [ ] Check the sentinel log for any anomalies
- [ ] Compare against a random baseline — is your agent actually learning?
- [ ] Inspect the final policy behavior with `multiverse sim preview`
- [ ] Save a run summary with `multiverse runs inspect`

---

## Congratulations!

You've completed the Multiverse learning path.

You can now:
- Train any algorithm on any verse
- Understand what Q-learning, SARSA, DQN, and PPO are actually doing
- Design reward functions that work
- Build custom environments
- Run reliable multi-seed experiments
- Use Multiverse's memory and safety systems

### Where to Go Next

- **Research**: Read `paper/` for the theoretical foundations
- **Production**: See `docs/SETUP.md` for deployment guidance
- **Extend**: Contribute new verses to `verses/`
- **Benchmark**: Run `python tools/benchmark_meta_stages.py` to measure your system